# LeetCode #1114: Print in Order

https://leetcode.com/problems/print-in-order/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin loop on shared flag | Wastes CPU |
| **Optimal: Semaphore ★** | Two `SemaphoreSlim(0,1)` gates | Blocks caller until predecessor signals |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Semaphore ★
Create two semaphores initially at 0. `first()` signals `sem1`; `second()` waits on `sem1` then signals `sem2`; `third()` waits on `sem2`. Each thread blocks until its predecessor completes — zero CPU waste.

**Constraints:**
* Exactly three threads call `first()`, `second()`, `third()` in some unknown order
* Output must always be `firstsecondthird`


## Solutions
### C#

In [ ]:
using System.Threading;

public class Foo
{
    private readonly SemaphoreSlim _sem1 = new SemaphoreSlim(0, 1);
    private readonly SemaphoreSlim _sem2 = new SemaphoreSlim(0, 1);

    public void First(Action printFirst)
    {
        printFirst();
        _sem1.Release();
    }

    public void Second(Action printSecond)
    {
        _sem1.Wait();
        printSecond();
        _sem2.Release();
    }

    public void Third(Action printThird)
    {
        _sem2.Wait();
        printThird();
    }
}

### Python

In [ ]:
import threading

class Foo:
    def __init__(self):
        self.sem1 = threading.Semaphore(0)
        self.sem2 = threading.Semaphore(0)

    def first(self, printFirst: 'Callable[[], None]') -> None:
        printFirst()
        self.sem1.release()

    def second(self, printSecond: 'Callable[[], None]') -> None:
        self.sem1.acquire()
        printSecond()
        self.sem2.release()

    def third(self, printThird: 'Callable[[], None]') -> None:
        self.sem2.acquire()
        printThird()

### Go

In [ ]:
package main

type Foo struct {
	sem1 chan struct{}
	sem2 chan struct{}
}

func NewFoo() *Foo {
	return &Foo{
		sem1: make(chan struct{}, 1),
		sem2: make(chan struct{}, 1),
	}
}

func (f *Foo) First(printFirst func()) {
	printFirst()
	f.sem1 <- struct{}{}
}

func (f *Foo) Second(printSecond func()) {
	<-f.sem1
	printSecond()
	f.sem2 <- struct{}{}
}

func (f *Foo) Third(printThird func()) {
	<-f.sem2
	printThird()
}

### Rust

In [ ]:
use std::sync::{Arc, Condvar, Mutex};

struct Foo {
    state: Mutex<u8>,
    cv: Condvar,
}

impl Foo {
    fn new() -> Arc<Self> {
        Arc::new(Foo { state: Mutex::new(0), cv: Condvar::new() })
    }

    fn first(&self, print_first: impl Fn()) {
        print_first();
        *self.state.lock().unwrap() = 1;
        self.cv.notify_all();
    }

    fn second(&self, print_second: impl Fn()) {
        let mut s = self.state.lock().unwrap();
        while *s < 1 { s = self.cv.wait(s).unwrap(); }
        print_second();
        *s = 2;
        self.cv.notify_all();
    }

    fn third(&self, print_third: impl Fn()) {
        let mut s = self.state.lock().unwrap();
        while *s < 2 { s = self.cv.wait(s).unwrap(); }
        print_third();
    }
}

## Concurrency Scenarios

1. **Threads arrive in order (1→2→3)**: `second` and `third` find the semaphores already signaled and proceed immediately without blocking.
2. **Thread 3 arrives first**: `third` blocks on `sem2`; `second` blocks on `sem1`; `first` runs, signals `sem1`, unblocking `second`, which then signals `sem2`.
3. **Thread 2 arrives before Thread 1**: `second` blocks on `sem1` until `first` completes and releases it.
4. **All three threads start simultaneously**: Only `first` can run; `second` and `third` block on their respective semaphores in the correct order.
5. **Semaphore count stays 0 until released**: No spurious wake-ups possible because `SemaphoreSlim` is not subject to spurious wakeups — only `Release()` unblocks a waiter.
